# Logistic Regression Titanic

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('../../../../data/titanic/train.csv')

In [ ]:
df.head()

In [ ]:
df_test = pd.read_csv('../../../../data/titanic/test.csv')

In [ ]:
df_test

In [ ]:
df.info()

In [ ]:
df_test.isnull().sum()

In [ ]:
df.info()


In [ ]:
df_test.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.corr(numeric_only=True)

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
## Data Cleaning
df.isnull().sum()

In [ ]:
df[df.isnull().any(axis=1)]

Cabin has 687 null values and age has 177
Imputing the Age and seeing the cabin column to see options in terms of available options

In [ ]:
df.info()

In [ ]:
sns.histplot(df['Cabin'])

In [ ]:
df.columns=df.columns.str.strip()
df.columns

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
df.hist(bins=50,figsize=(20,15))

In [ ]:
sns.set_theme(style="darkgrid")

df.hist(bins=50, figsize=(20, 15))
plt.show()

In [ ]:
plt.figure(figsize=(15, 7))
sns.heatmap(df.isnull(),yticklabels=False,cbar=False,cmap='coolwarm')

In [ ]:
plt.figure(figsize=(10, 7))
sns.set_style('whitegrid')
sns.countplot(x='Survived',data=df,palette='RdBu_r')

survived => 0-no 1-yes

In [ ]:
plt.figure(figsize=(10, 7))
sns.set_style('whitegrid')
sns.countplot(x='Survived',hue='Sex',data=df,palette='RdBu_r')

In [ ]:
plt.figure(figsize=(10, 7))
sns.set_style('whitegrid')
sns.countplot(x='Survived',hue='Pclass',data=df,palette='rainbow')

In [ ]:
sns.distplot(df['Age'].dropna(),kde=False,color='darkred',bins=30)

In [ ]:
sns.countplot(x='SibSp',data=df)

In [ ]:
df['Fare'].hist(color='green',bins=40,figsize=(8,4))

In [ ]:
plt.figure(figsize=(12, 7))
sns.boxplot(x='Pclass',y='Age',data=df,palette='winter')

In [ ]:
map_mean = df.groupby('Pclass')['Age'].mean()
map_mean

In [ ]:
def impute_age(cols):
    Age = cols[0]
    PClass = cols[1]

    if pd.isnull(Age):
        if PClass == 1:
            return 38
        elif PClass == 1:
            return 30
        else:
            return 25
    else:
        return Age

In [ ]:
df['Age'] = df[['Age','Pclass']].apply(impute_age,axis=1)

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
plt.figure(figsize=(15, 7))
sns.heatmap(df.isnull(),yticklabels=False,cbar=False,cmap='coolwarm')

In [ ]:
df['Cabin'].unique()

We can not impute any values here even one hot encoding will fail and if we were to put a 0 for missing and any other number it will just act as an unnecessary feature

In [ ]:
df.head()

In [ ]:
df.drop('Cabin', axis=1, inplace=True)
df.head()

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df['Embarked'].unique()

In [ ]:
df=df.drop(df[df['Embarked'].isnull()].index, axis = 0)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df['Sex'].unique()

In [ ]:
sex = pd.get_dummies(df['Sex'],drop_first=True, dtype=int)
embark = pd.get_dummies(df['Embarked'],drop_first=False, dtype=int)
sex

In [ ]:
embark

In [ ]:
df['Ticket'].unique()

In [ ]:
df.drop(['Sex','Embarked','Name','Ticket'],axis=1,inplace=True)

In [ ]:
df = pd.concat([df,sex,embark], axis =1)
df

In [ ]:
df.rename(columns={'male': 'sex'}, inplace=True)


1-Male, 0 - Female

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.to_csv(
    "../../../../data/titanic/cleaned_data_train_titanic.csv",
    index=False,
    encoding="utf-8",

)


Now we can train

In [ ]:
X = df.drop(['Survived'], axis=1)
y = df['Survived']


In [ ]:
X

In [ ]:
y

In [ ]:
y.value_counts()

In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    test_size=0.25,
                                                    random_state=101)

In [ ]:
X_train.shape

In [ ]:
class LogisticRegressionSelf:
    def __init__(self,
            learning_rate=0.01,
            epochs=5000,
            batch_size=None,
            early_stopping=True,
            patience=15):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.early_stopping = early_stopping
        self.patience = patience
        self.w_=None
        self.b_=0
        self.losses = []

    def accuracy(self, y,y_pred):
        return np.mean(y==y_pred)

    def log_loss(self, y, y_pred):
        ep = 1e-15
        y_pred = np.clip(y_pred, ep, 1-ep)
        return -np.mean(y*np.log(y_pred)+(1-y)*np.log(1-y_pred))

    def sigmoid(self, z):
        return 1/(1+np.exp(-z))

    def fit(self, X, y):
        return self._fit_internal(X, y)

    def _fit_internal(self, X, y, auto_plot=True, verbose=True):
        X = np.array(X)
        y = np.array(y)
        self.X_raw = X if X.ndim > 1 else X.reshape(-1, 1)
        self.y_raw = y
        n_samples, n_features = X.shape
        if len(y)!=n_samples:
            raise ValueError("X and Y doesn't have same number of values")
        # self.w_=np.zeros(n_features)
        self.w_ = np.random.randn(n_features) * 0.007  # Small random values

        self.b_=0

        batch_size = self.batch_size or n_samples
        best_loss = float('inf')
        patience_counter = 0
        for epoch in range(self.epochs):
            indices = np.random.permutation(n_samples)
            X_shuffled =X[indices]
            y_shuffled =y[indices]
            for i in range(0,n_samples,batch_size):
                X_batch = X_shuffled[i:i+batch_size]
                y_batch = y_shuffled[i:i+batch_size]
                y_pred = self.sigmoid((X_batch@self.w_)+self.b_)

                dw = (1/len(y_batch)) * (X_batch.T @ (y_pred - y_batch))
                db = (1/len(y_batch)) * (np.sum(y_pred - y_batch))

                self.w_ -= self.learning_rate*dw
                self.b_ -= self.learning_rate*db
            full_pred = self.sigmoid(X @ self.w_ + self.b_)
            loss = self.log_loss(y,full_pred)
             # loss = self.mse(y,full_pred)
            regularization_strength = 0.007  # Can be adjusted
            loss += regularization_strength * np.sum(self.w_**2)  # L2 regularization term

            self.losses.append(loss)
            if(self.early_stopping):
                if(loss<best_loss):
                    best_loss = loss
                    patience_counter=0
                else:
                    patience_counter+=1
                if patience_counter>=self.patience:
                    print(f'Early Stopping at epoch: {epoch}')
                    break
            if verbose and epoch % 100 == 0:
                print(f'Epoch {epoch}, Loss: {loss:.6f}')
        if(verbose): print("Training Completed")
        if auto_plot:
            self.plot_all_diagnostics(self.X_raw, self.y_raw)

    def predict(self, X, threshold=0.33):
        if self.w_ is None:
            raise ValueError("Model has not been trained yet")
        X=np.array(X)
        probs = self.sigmoid(X@self.w_ + self.b_)
        return (probs>=threshold).astype(int)

    def predict_proba(self, X):
        if self.w_ is None:
                raise ValueError("Model has not been trained yet")
        X=np.array(X)
        return self.sigmoid(X @ self.w_ + self.b_)


    def plot_all_diagnostics(self, X, y):
        """ Plot all diagnostics for training evaluation """
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # Plot 1: Learning Curve (Loss over epochs)
        axes[0].plot(self.losses, color='navy')
        axes[0].set_title('Learning Curve (Loss)')
        axes[0].set_xlabel('Epochs')
        axes[0].set_ylabel('Log-Loss')
        axes[0].grid(True, alpha=0.3)

        # Plot 2: Actual vs Predicted
        y_pred = self.predict(X)
        axes[1].scatter(y, y_pred, alpha=0.5, color='teal')
        min_val, max_val = min(y.min(), y_pred.min()), max(y.max(), y_pred.max())
        axes[1].plot([min_val, max_val], [min_val, max_val], 'r--')
        axes[1].set_title(f'Actual vs Pred (Accuracy: {self.accuracy(y, y_pred):.2f})')
        axes[1].set_xlabel('Actual')
        axes[1].set_ylabel('Predicted')

        # Plot 3: Residuals
        residuals = y - y_pred
        axes[2].scatter(y_pred, residuals, alpha=0.5, color='purple')
        axes[2].axhline(0, color='red', linestyle='--')
        axes[2].set_title('Residuals (Errors)')
        axes[2].set_xlabel('Predicted')

        plt.tight_layout()
        plt.show()



In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
lr = LogisticRegressionSelf()
lr.fit(X_train_scaled, y_train)

In [ ]:
y_pred = lr.predict(X_test_scaled)

In [ ]:
acc = lr.accuracy(y=y_test, y_pred=y_pred)
acc

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

cm = confusion_matrix(y_test, y_pred)
cr = classification_report(y_test, y_pred)

print("Confusion Matrix:\n", cm)
print("Classification Report:\n", cr)


In [ ]:
probs = lr.predict_proba(X_test_scaled)
plt.hist(probs, bins=20)
plt.show()


In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lrO = LogisticRegression(verbose=2)
lrO.fit(X_train_scaled, y_train)

In [ ]:
y_pred = lrO.predict(X_test_scaled)
acc = lr.accuracy(y=y_test, y_pred=y_pred)
acc

In [ ]:
y_probs = lr.predict_proba(X_test_scaled)
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
J = tpr - fpr
best_idx = np.argmax(J)
best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)
print("TPR:", tpr[best_idx])
print("FPR:", fpr[best_idx])
lr.predict(X_test, threshold=best_threshold)


In [ ]:
y_pred = lr.predict(X_test_scaled, threshold=0.2784)
acc = lr.accuracy(y=y_test, y_pred=y_pred)
print(f'accuracy score: {acc}')
from sklearn.metrics import classification_report, confusion_matrix

cm = confusion_matrix(y_test, y_pred)
cr = classification_report(y_test, y_pred)

print("Confusion Matrix:\n", cm)
print("Classification Report:\n", cr)
# optimal ONE!!!!!!!!!!!!


In [ ]:
y_probs_sk = lrO.predict_proba(X_test_scaled)[:, 1]

fpr_sk, tpr_sk, _ = roc_curve(y_test, y_probs_sk)
auc_sk = auc(fpr_sk, tpr_sk)

print(auc_sk)


In [ ]:
probs = lr.predict_proba(X_test_scaled)
for thresh in [0.3, 0.35, 0.4, 0.45, 0.5]:
    preds = (probs >= thresh).astype(int)
    print(f"Threshold: {thresh}")
    print(classification_report(y_test, preds))


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
probs = lr.predict_proba(X_test_scaled)

# Define thresholds to test
thresholds = np.linspace(0.1, 0.5, 41)  # 0.1 to 0.5 in 0.01 steps

precisions = []
recalls = []
f1s = []

# Calculate metrics for each threshold
for t in thresholds:
    preds = (probs >= t).astype(int)
    precisions.append(precision_score(y_test, preds))
    recalls.append(recall_score(y_test, preds))
    f1s.append(f1_score(y_test, preds))

# Plot
plt.figure(figsize=(10,6))
plt.plot(thresholds, precisions, label='Precision', marker='o')
plt.plot(thresholds, recalls, label='Recall', marker='s')
plt.plot(thresholds, f1s, label='F1-score', marker='^')
plt.axvline(x=0.33, color='red', linestyle='--', label='Chosen threshold = 0.33')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision, Recall, F1 vs Threshold (Titanic)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Classification Metrics Cheat Sheet

### 1️⃣ Confusion Matrix

|                | Pred 0 | Pred 1 |
|----------------|--------|--------|
| **Actual 0**   | TN     | FP     |
| **Actual 1**   | FN     | TP     |

Where:

- **TP** = True Positive → correctly predicted positive (e.g., survivor)  
- **TN** = True Negative → correctly predicted negative (e.g., non-survivor)  
- **FP** = False Positive → predicted positive but actually negative  
- **FN** = False Negative → predicted negative but actually positive  

---

### 2️⃣ Accuracy

\[
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
\]

- **Meaning:** Fraction of all correct predictions.  
- **Limitation:** Can be misleading for imbalanced datasets.

---

### 3️⃣ Precision (Positive Predictive Value)

\[
\text{Precision} = \frac{TP}{TP + FP}
\]

- **Meaning:** "When I predict positive, how often am I correct?"  
- High precision → few false positives.

---

### 4️⃣ Recall (Sensitivity / True Positive Rate, TPR)

\[
\text{Recall} = \frac{TP}{TP + FN}
\]

- **Meaning:** "Of all actual positives, how many did I catch?"  
- High recall → few false negatives.

---

### 5️⃣ F1-Score

\[
F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
\]

- **Meaning:** Harmonic mean of precision and recall.  
- Balances precision vs recall.  
- Useful when both false positives and false negatives matter.

---

### 6️⃣ True Negative Rate (Specificity)

\[
\text{TNR (Specificity)} = \frac{TN}{TN + FP}
\]

- **Meaning:** Fraction of actual negatives correctly predicted.

---

### 7️⃣ False Positive Rate (FPR)

\[
\text{FPR} = \frac{FP}{FP + TN} = 1 - \text{Specificity}
\]

- **Meaning:** Fraction of negatives incorrectly predicted as positive.

---

## TL;DR in words

- **Precision:** Accuracy of positive predictions  
- **Recall:** Coverage of actual positives  
- **F1-score:** Balance between precision and recall  
- **Accuracy:** Overall correctness  
- **TNR / Specificity:** Correctness on negatives  
- **FPR:** Mistakes on negatives
